<a href="https://colab.research.google.com/github/bhoomiy/My_AIML_Journey/blob/main/CNN_for_CIFAR10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torchvision
from torchvision.datasets import CIFAR10
import torch.optim as optim

In [3]:
#Datasets and  Dataloaders
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

#data-->scale(0,1)-->normalize(-1,1)
transform=transforms.Compose([
    transforms.ToTensor(), #converts them to tensors and scales them
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

trainset=CIFAR10(root="./data",train=True,download=True,transform=transform)
testset=CIFAR10(root="./data",train=False,download=True,transform=transform)

In [4]:
trainloader=DataLoader(trainset,batch_size=64,shuffle=True)
testloader=DataLoader(testset,batch_size=64)

### BUILD THE CNN

In [10]:
class CNN(nn.Module):
  def __init__(self):
    super(CNN,self).__init__()

    self.conv_layers=nn.Sequential(
        nn.Conv2d(3,32,kernel_size=3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2),#kernel size=2,stride=2

        nn.Conv2d(32,64,kernel_size=3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2),#kernel size=2,stride=2

        nn.Conv2d(64,128,kernel_size=3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2),#kernel size=2,stride=2

    )

    self.fc_layers=nn.Sequential(
        nn.Linear(4*4*128,256),
        nn.ReLU(),

        nn.Linear(256,10)
    )

  def forward(self,x):
    x=self.conv_layers(x)
    x=x.view(x.size(0),-1) #flattening
    x=self.fc_layers(x)

    return x

In [11]:
model=CNN()

In [12]:
criteria=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters())

### TRAIN THE CNN

In [14]:
epochs=10
for epoch in range(epochs):
  training_loss=0.0

  for images,labels in trainloader:
    optimizer.zero_grad()

    output=model.forward(images) #FP
    loss=criteria(output,labels) #losss fnx
    loss.backward() #BP
    optimizer.step() #update parameters

    training_loss+=loss.item()

  print(f"epoch={epoch+1}/{epochs} &loss={training_loss/len(trainloader)}")

epoch=1/10 &loss=0.872333158388772
epoch=2/10 &loss=0.7055107527376746
epoch=3/10 &loss=0.583429779199993
epoch=4/10 &loss=0.4776778661495889
epoch=5/10 &loss=0.3790975094241712
epoch=6/10 &loss=0.2986862361145294
epoch=7/10 &loss=0.23413957698304025
epoch=8/10 &loss=0.17572070557214414
epoch=9/10 &loss=0.13501591061758797
epoch=10/10 &loss=0.11549527895853609


### EVALUATE

In [16]:

correct_labels=0
total_labels=0.0
model.eval()
with torch.no_grad():
  for images,labels in testloader:
    output=model.forward(images)
    _,predicted=torch.max(output,1)

    correct_labels += (predicted == labels).sum().item()
    total_labels += labels.size(0)

print(f"accuracy = {correct_labels / total_labels * 100}")


accuracy = 75.71
